<a href="https://colab.research.google.com/github/thetireddude/self-distillation-lab/blob/main/Apple_Paper_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [293]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes evaluate
!pip install -q human-eval

[GPU] 0, 6170, 23034
[GPU] 0, 6170, 23034
[GPU] 0, 6170, 23034


In [294]:
import os
from google.colab import userdata

# Automatically retrieves your token and authenticates the environment
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [295]:
!hf auth login

User is already logged in. Use `hf auth login --force` to force re-login.


In [296]:
!hf auth whoami

✓ Logged in
  user: tiredk


In [297]:
!nvidia-smi

Fri Jul  3 03:44:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P0             28W /   72W |    6170MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [298]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [299]:
EXPERIMENT_NAME = "temp0.5_topk20_topp0.8_mbpp_full_code_extracted_run02"

BASE_DIR = f"/content/drive/MyDrive/apple_ssd_experiment/formatted_prompt/{EXPERIMENT_NAME}"

os.makedirs(BASE_DIR, exist_ok=True)

In [300]:
import subprocess, threading, time

def gpu_monitor(interval=30):
    while True:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True
        )
        print(f"[GPU] {result.stdout.strip()}")
        time.sleep(interval)

threading.Thread(target=gpu_monitor, daemon=True).start()

In [301]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
import torch

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

[GPU] 0, 6170, 23034


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [302]:
import re, json, os, tempfile, subprocess, textwrap, torch
from datasets import load_dataset
from tqdm import tqdm
from torch.utils.data import DataLoader

humaneval = load_dataset("openai/openai_humaneval", split="test")

def extract_code(text):
    # Remove markdown fences if present
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    if match:
        text = match.group(1)

    # Stop before model-written tests / explanations
    stop_markers = [
        "\nif __name__",
        "\ndef test_",
        "\n# Check",
        "\ncheck_solution",
        "\nprint(",
        "\n```",
        "\nThis Python",
        "\nExplanation",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    return text.rstrip()

def batched_generate_completions(
    model,
    tokenizer,
    problems,
    batch_size=32,
    max_new_tokens=256,
    save_path=None
):
    model.eval()
    completions = {}

    if save_path and os.path.exists(save_path):
        with open(save_path, "r") as f:
            completions = json.load(f)
        print(f"Loaded {len(completions)} existing completions")

    remaining = [p for p in problems if p["task_id"] not in completions]
    print(f"Remaining completions to generate: {len(remaining)}")

    loader = DataLoader(
        remaining,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=lambda batch: batch
    )

    for batch in tqdm(loader, desc="Generating HumanEval completions"):
        prompts = [p["prompt"] for p in batch]
        task_ids = [p["task_id"] for p in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.batch_decode(
            outputs[:, input_len:],
            skip_special_tokens=True
        )

        for task_id, text in zip(task_ids, decoded):
            completions[task_id] = extract_code(text)

        if save_path:
            with open(save_path, "w") as f:
                json.dump(completions, f, indent=2)

    return completions

def run_humaneval_test(problem, completion, timeout=10):
    code = (
        problem["prompt"]
        + "\n"
        + completion
        + "\n"
        + problem["test"]
        + f"\ncheck({problem['entry_point']})"
    )

    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        path = f.name

    try:
        result = subprocess.run(
            ["python", path],
            capture_output=True,
            text=True,
            timeout=timeout
        )
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.remove(path)

def evaluate_humaneval_fast(
    model,
    tokenizer,
    results_path,
    completions_path,
    batch_size=8,
    max_new_tokens=256,
    timeout=10
):
    problems = list(humaneval)

    completions = batched_generate_completions(
        model=model,
        tokenizer=tokenizer,
        problems=problems,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
        save_path=completions_path
    )

    results = []
    passed = 0

    for problem in tqdm(problems, desc="Running HumanEval tests"):
        completion = completions[problem["task_id"]]
        ok = run_humaneval_test(problem, completion, timeout=timeout)

        passed += int(ok)

        results.append({
            "task_id": problem["task_id"],
            "passed": ok,
            "completion": completion
        })

    score = passed / len(problems)

    with open(results_path, "w") as f:
        json.dump({
            "pass_at_1": score,
            "num_passed": passed,
            "num_total": len(problems),
            "results": results
        }, f, indent=2)

    print(f"HumanEval pass@1: {score:.3f} ({passed}/{len(problems)})")

    return score

In [303]:
base_eval_path = f"{BASE_DIR}/humaneval_base_results.json"
base_completions_path = f"{BASE_DIR}/humaneval_base_completions.json"

base_score = evaluate_humaneval_fast(
    model=model,
    tokenizer=tokenizer,
    results_path=base_eval_path,
    completions_path=base_completions_path,
    batch_size=32,
    max_new_tokens=256,
    timeout=10
)

Remaining completions to generate: 164


Generating HumanEval completions:   0%|          | 0/6 [00:00<?, ?it/s]

[GPU] 100, 6170, 23034
[GPU] 41, 6170, 23034
[GPU] 42, 6170, 23034
[GPU] 47, 6170, 23034
[GPU] 47, 6170, 23034
[GPU] 46, 6170, 23034
[GPU] 47, 6170, 23034


Generating HumanEval completions:  17%|█▋        | 1/6 [00:28<02:21, 28.33s/it]

[GPU] 41, 6170, 23034
[GPU] 43, 6170, 23034
[GPU] 43, 6170, 23034
[GPU] 53, 6170, 23034
[GPU] 53, 6170, 23034
[GPU] 55, 6170, 23034
[GPU] 57, 6170, 23034


...

[GPU] 0, 6170, 23034


Running HumanEval tests:  94%|█████████▍| 154/164 [00:09<00:00, 16.55it/s]

[GPU] 0, 6170, 23034


Running HumanEval tests: 100%|██████████| 164/164 [00:09<00:00, 16.45it/s]

HumanEval pass@1: 0.427 (70/164)


In [304]:
import json

completions_path = f"{BASE_DIR}/humaneval_base_completions.json"

with open(completions_path, "r") as f:
    completions = json.load(f)

print(f"Loaded {len(completions)} completions")

for task_id, completion in list(completions.items())[:3]:
    print("=" * 80)
    print("TASK:", task_id)
    print(completion[:2000])

Loaded 164 completions
TASK: HumanEval/0
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False
TASK: HumanEval/1
    paren_list = []
    current_group = ''
    for char in paren_string:
        if char == '(':
            current_group += char
        elif char == ')':
            current_group += char
            paren_list.append(current_group)
            current_group = ''

    return paren_list
TASK: HumanEval/2
    return number - int(number)


In [305]:
empty_or_short = {
    task_id: completion
    for task_id, completion in completions.items()
    if len(completion.strip()) < 20
}

print("Very short completions:", len(empty_or_short))
print(list(empty_or_short.items())[:5])

Very short completions: 4
[('HumanEval/75', '    # Your code here'), ('HumanEval/127', '    # your code here'), ('HumanEval/150', '    # Your code here'), ('HumanEval/153', '    # Your code here')]


In [306]:
from datasets import load_dataset

dataset = load_dataset("claudios/google-research-datasets__mbpp", "full", split="train+validation+test")

In [307]:
len(dataset)

964

In [308]:
MBPP_PROMPT = """You will be given a programming problem and will generate a correct Python function that matches the specification and passes all tests.

You will NOT return anything except for the function implementation.

Question:
{question}

Return ONLY valid Python code, do NOT provide explanations, examples, or markdown. Do NOT repeat the problem statement or write any text before or after the code. Complete only the requested function.

Enclose your code within delimiters exactly as follows. The final solution should be formatted as:
```python # YOUR CODE HERE ```"""


def make_prompt(example):
    return MBPP_PROMPT.format(question=example["text"])

prompts = [make_prompt(x) for x in dataset]


In [309]:
prompts

['You will be given a programming problem and will generate a correct Python function that matches the specification and passes all tests.\n\nYou will NOT return anything except for the function implementation.\n\nQuestion:\nWrite a function to find the longest chain which can be formed from the given set of pairs.\n\nReturn ONLY valid Python code, do NOT provide explanations, examples, or markdown. Do NOT repeat the problem statement or write any text before or after the code. Complete only the requested function.\n\nEnclose your code within delimiters exactly as follows. The final solution should be formatted as: \n```python # YOUR CODE HERE ```',
 'You will be given a programming problem and will generate a correct Python function that matches the specification and passes all tests.\n\nYou will NOT return anything except for the function implementation.\n\nQuestion:\nWrite a python function to find the first repeated character in a given string.\n\nReturn ONLY valid Python code, do 

In [310]:
import json, os, torch
from tqdm import tqdm
from torch.utils.data import DataLoader

output_path = f"{BASE_DIR}/ssd_data.jsonl"
batch_size = 64  # start with 4 or 8 on Colab T4; increase if memory allows

# Load completed prompts
generated = {}
if os.path.exists(output_path):
    with open(output_path, "r") as f:
        for line in f:
            row = json.loads(line)
            generated[row["prompt"]] = row

remaining_prompts = [p for p in prompts if p not in generated]

print(f"Already generated: {len(generated)}")
print(f"Remaining: {len(remaining_prompts)}")

prompt_loader = DataLoader(
    remaining_prompts,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

model.eval()

with open(output_path, "a") as f:
    for batch_prompts in tqdm(prompt_loader):
        inputs = tokenizer(
            list(batch_prompts),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.5,
                top_k=20,
                top_p=0.8,
                pad_token_id=tokenizer.eos_token_id
            )

        input_len = inputs["input_ids"].shape[1]
        completions = tokenizer.batch_decode(
            outputs[:, input_len:],
            skip_special_tokens=True
        )

        for prompt, completion in zip(batch_prompts, completions):
            row = {
                "prompt": prompt,
                "completion": completion
            }
            f.write(json.dumps(row) + "\n")
            f.flush()

Already generated: 0
Remaining: 964


  0%|          | 0/16 [00:00<?, ?it/s]

[GPU] 53, 6170, 23034
[GPU] 56, 6170, 23034
[GPU] 61, 6170, 23034
[GPU] 61, 6170, 23034
[GPU] 67, 6170, 23034
[GPU] 68, 6170, 23034
[GPU] 72, 6170, 23034
[GPU] 85, 6286, 23034
[GPU] 87, 6402, 23034
[GPU] 91, 6518, 23034
[GPU] 91, 6518, 23034
[GPU] 88, 6518, 23034
[GPU] 100, 6634, 23034
[GPU] 97, 6750, 23034


  6%|▋         | 1/16 [01:02<15:37, 62.53s/it]

[GPU] 54, 6750, 23034
[GPU] 55, 6750, 23034
[GPU] 66, 6750, 23034
[GPU] 66, 6750, 23034
[GPU] 67, 6750, 23034
[GPU] 73, 6750, 23034
[GPU] 77, 6750, 23034
[GPU] 85, 6750, 23034
[GPU] 86, 6750, 23034
[GPU] 93, 6750, 23034
[GPU] 93, 6750, 23034
[GPU] 94, 6750, 23034
[GPU] 96, 6750, 23034
[GPU] 97, 6750, 23034


...

 88%|████████▊ | 14/16 [14:36<02:05, 62.77s/it]

[GPU] 52, 6866, 23034
[GPU] 57, 6866, 23034
[GPU] 57, 6866, 23034
[GPU] 59, 6866, 23034
[GPU] 65, 6866, 23034
[GPU] 72, 6866, 23034
[GPU] 77, 6866, 23034
[GPU] 78, 6866, 23034
[GPU] 85, 6866, 23034
[GPU] 85, 6866, 23034
[GPU] 87, 6866, 23034
[GPU] 91, 6866, 23034
[GPU] 97, 6866, 23034
[GPU] 97, 6866, 23034
[GPU] 97, 6866, 23034


 94%|█████████▍| 15/16 [15:39<01:02, 62.67s/it]

[GPU] 32, 6866, 23034
[GPU] 32, 6866, 23034
[GPU] 32, 6866, 23034
[GPU] 32, 6866, 23034
[GPU] 32, 6866, 23034
[GPU] 33, 6866, 23034
[GPU] 33, 6866, 23034
[GPU] 33, 6866, 23034
[GPU] 33, 6866, 23034
[GPU] 34, 6866, 23034
[GPU] 33, 6874, 23034
[GPU] 32, 6874, 23034


100%|██████████| 16/16 [16:34<00:00, 62.17s/it]


In [311]:
from datasets import Dataset
import json

output_path = f"{BASE_DIR}/ssd_data.jsonl"

ssd_data = []
with open(output_path, "r") as f:
    for line in f:
        ssd_data.append(json.loads(line))

print(f"Loaded {len(ssd_data)} SSD examples")

train_dataset = Dataset.from_list(ssd_data)

def extract_python_code(completion):
    match = re.search(r"```python\s*(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1).strip()

    match = re.search(r"```\s*(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1).strip()

    text = completion.strip()

    stop_markers = [
        "\nNote:",
        "\nExplanation:",
        "\nTest Case",
        "\nTest Cases",
        "\nExample",
        "\nExamples",
        "\nIf the input",
        "\nThe output",
        "\nTo solve",
        "\nThis function",
        "\nPlease",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    return text

def format_example(example):
    code = extract_python_code(example["completion"])
    return {
        "text": example["prompt"] + "\n" + code
    }

train_dataset = train_dataset.map(
    format_example,
    remove_columns=["prompt", "completion"]
)


Loaded 964 SSD examples


Map:   0%|          | 0/964 [00:00<?, ? examples/s]

In [312]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

In [313]:
train_dataset

Dataset({
    features: ['text'],
    num_rows: 964
})

In [314]:
train_dataset.column_names

['text']

In [315]:
train_dataset[0].keys()

dict_keys(['text'])

In [316]:
train_dataset[0]["text"]

'You will be given a programming problem and will generate a correct Python function that matches the specification and passes all tests.\n\nYou will NOT return anything except for the function implementation.\n\nQuestion:\nWrite a function to find the longest chain which can be formed from the given set of pairs.\n\nReturn ONLY valid Python code, do NOT provide explanations, examples, or markdown. Do NOT repeat the problem statement or write any text before or after the code. Complete only the requested function.\n\nEnclose your code within delimiters exactly as follows. The final solution should be formatted as: \n```python # YOUR CODE HERE ```\ndef longest_chain(pairs):\n    pairs.sort(key=lambda x: x[1])\n    dp = [1] * len(pairs)\n    for i in range(1, len(pairs)):\n        for j in range(i):\n            if pairs[i][0] > pairs[j][1]:\n                dp[i] = max(dp[i], dp[j] + 1)\n    return max(dp)'

In [317]:
train_dataset[0]

{'text': 'You will be given a programming problem and will generate a correct Python function that matches the specification and passes all tests.\n\nYou will NOT return anything except for the function implementation.\n\nQuestion:\nWrite a function to find the longest chain which can be formed from the given set of pairs.\n\nReturn ONLY valid Python code, do NOT provide explanations, examples, or markdown. Do NOT repeat the problem statement or write any text before or after the code. Complete only the requested function.\n\nEnclose your code within delimiters exactly as follows. The final solution should be formatted as: \n```python # YOUR CODE HERE ```\ndef longest_chain(pairs):\n    pairs.sort(key=lambda x: x[1])\n    dp = [1] * len(pairs)\n    for i in range(1, len(pairs)):\n        for j in range(i):\n            if pairs[i][0] > pairs[j][1]:\n                dp[i] = max(dp[i], dp[j] + 1)\n    return max(dp)'}

In [318]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=f"{BASE_DIR}/checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    max_length=1024,
    packing=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    args=training_args
)

[GPU] 0, 6874, 23034


Adding EOS to train dataset:   0%|          | 0/964 [00:00<?, ? examples/s]

[GPU] 0, 6874, 23034


Tokenizing train dataset:   0%|          | 0/964 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/964 [00:00<?, ? examples/s]

In [319]:
# use if starting fresh
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


[GPU] 19, 6118, 23034
[GPU] 19, 6118, 23034
[GPU] 33, 6128, 23034


Step,Training Loss
10,2.103593
20,0.604682
30,0.320997
40,0.308158
50,0.251741
60,0.228952
70,0.209814
80,0.204873
90,0.210274
100,0.211401


[GPU] 22, 6128, 23034
[GPU] 54, 6128, 23034
[GPU] 22, 6128, 23034
[GPU] 34, 6128, 23034
[GPU] 28, 6128, 23034
[GPU] 28, 6128, 23034
[GPU] 22, 6128, 23034
[GPU] 20, 6128, 23034
[GPU] 61, 6128, 23034
[GPU] 20, 6128, 23034
[GPU] 55, 6128, 23034
[GPU] 21, 6128, 23034
[GPU] 21, 6128, 23034
[GPU] 20, 6128, 23034
[GPU] 35, 6128, 23034
[GPU] 21, 6128, 23034
[GPU] 34, 6128, 23034
[GPU] 20, 6128, 23034
[GPU] 22, 6130, 23034
[GPU] 22, 6130, 23034
[GPU] 59, 6130, 23034
[GPU] 33, 6130, 23034
[GPU] 21, 6130, 23034
[GPU] 57, 6132, 23034
[GPU] 18, 6132, 23034
[GPU] 21, 6132, 23034
[GPU] 21, 6132, 23034
[GPU] 27, 6132, 23034
[GPU] 21, 6132, 23034
[GPU] 31, 6132, 23034
[GPU] 23, 6132, 23034
[GPU] 57, 6132, 23034
[GPU] 55, 6132, 23034
[GPU] 55, 6132, 23034
[GPU] 21, 6132, 23034
[GPU] 36, 6132, 23034
[GPU] 23, 6132, 23034
[GPU] 68, 6132, 23034
[GPU] 30, 6132, 23034
[GPU] 54, 6136, 23034
[GPU] 54, 6136, 23034
[GPU] 20, 6136, 23034
[GPU] 16, 6136, 23034
[GPU] 25, 6136, 23034
[GPU] 18, 6136, 23034
[GPU] 54, 

TrainOutput(global_step=121, training_loss=0.42105311795699696, metrics={'train_runtime': 534.4256, 'train_samples_per_second': 1.804, 'train_steps_per_second': 0.226, 'total_flos': 1167334168888320.0, 'train_loss': 0.42105311795699696, 'entropy': 0.1466426532715559, 'num_tokens': 147985.0, 'mean_token_accuracy': 0.9577280282974243, 'epoch': 1.0})

In [320]:
# use if progress was interrupted

# trainer.train(resume_from_checkpoint=True)

In [321]:
trainer.save_model(f"{BASE_DIR}/ssd-qwen-coder-1.5b-lora")

In [322]:
from peft import PeftModel

adapter_path = f"{BASE_DIR}/ssd-qwen-coder-1.5b-lora"

base_for_eval = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

ssd_model = PeftModel.from_pretrained(
    base_for_eval,
    adapter_path
)

ssd_model.eval()

ssd_eval_path = f"{BASE_DIR}/humaneval_ssd_results.json"
ssd_completions_path = f"{BASE_DIR}/humaneval_ssd_completions.json"

ssd_score = evaluate_humaneval_fast(
    model=ssd_model,
    tokenizer=tokenizer,
    results_path=ssd_eval_path,
    completions_path=ssd_completions_path,
    batch_size=32,
    max_new_tokens=256,
    timeout=10
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[GPU] 35, 6136, 23034
Remaining completions to generate: 164


Generating HumanEval completions:   0%|          | 0/6 [00:00<?, ?it/s]

[GPU] 32, 6136, 23034
[GPU] 30, 6136, 23034
[GPU] 32, 6136, 23034
[GPU] 35, 6136, 23034
[GPU] 37, 6136, 23034
[GPU] 38, 6136, 23034
[GPU] 39, 6136, 23034
[GPU] 43, 6136, 23034
[GPU] 43, 6136, 23034


...

[GPU] 0, 6170, 23034


Running HumanEval tests:  67%|██████▋   | 110/164 [00:06<00:03, 16.62it/s]

[GPU] 0, 6170, 23034


Running HumanEval tests: 100%|██████████| 164/164 [00:09<00:00, 16.48it/s]

HumanEval pass@1: 0.451 (74/164)


In [323]:
print("Base HumanEval pass@1:", base_score)
print("SSD HumanEval pass@1:", ssd_score)
print("Delta:", ssd_score - base_score)

Base HumanEval pass@1: 0.4268292682926829
SSD HumanEval pass@1: 0.45121951219512196
Delta: 0.024390243902439046


In [324]:
# FOR DEBUGGING:

import json

with open(f"{BASE_DIR}/ssd_data.jsonl") as f:
    examples = [json.loads(next(f)) for _ in range(20)]

for ex in examples[:5]:
    print(ex["completion"][:1000])
    print("=" * 80)




```python
def longest_chain(pairs):
    pairs.sort(key=lambda x: x[1])
    dp = [1] * len(pairs)
    for i in range(1, len(pairs)):
        for j in range(i):
            if pairs[i][0] > pairs[j][1]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)
```

```python
# Test cases
print(longest_chain([(3, 4), (2, 3), (1, 2)]))  # Output: 3
print(longest_chain([(5, 7), (6, 8), (9, 10), (11, 12)]))  # Output: 4
print(longest_chain([(1, 2), (2, 3), (3, 4), (4, 5)]))  # Output: 5
``` ```python
# Your code here
``` ```python
# Test cases
print(longest_chain([(3, 4), (2, 3), (1, 2)]))  # Output: 3
print(longest_chain([(5, 7), (6, 8), (9, 10), (11, 12)]))  # Output: 4
print(longest_chain([(1, 2), (2, 3), (3, 4), (4, 5)]))  # Output: 5
``` ```python
# Your code here
``` ```python
# Test cases
print(longest_chain([(3, 4), (2, 3), (1, 2)]))  # Output: 3
print(longest_chain([(5, 7), (6, 8), (9, 10), (11, 12)]))  # Output: 4
print(longest_chain([(1, 2), (2, 3), (3, 4), (4, 5)]))  # 

In [325]:
# FOR DEBUGGING

import json, random

ssd_path = f"{BASE_DIR}/ssd_data.jsonl"

with open(ssd_path, "r") as f:
    ssd_examples = [json.loads(line) for line in f]

sample = random.sample(ssd_examples, min(50, len(ssd_examples)))

for i, ex in enumerate(sample):
    print("=" * 100)
    print("EXAMPLE", i)
    print("PROMPT:")
    print(ex["prompt"][:500])
    print("\nCOMPLETION:")
    print(ex["completion"][:1500])

EXAMPLE 0
PROMPT:
You will be given a programming problem and will generate a correct Python function that matches the specification and passes all tests.

You will NOT return anything except for the function implementation.

Question:
Write a python function to check whether the elements in a list are same or not.

Return ONLY valid Python code, do NOT provide explanations, examples, or markdown. Do NOT repeat the problem statement or write any text before or after the code. Complete only the requested function.

COMPLETION:



```python
def is_same_elements(lst):
    if len(lst) == 0:
        return True
    first_element = lst[0]
    for element in lst:
        if element != first_element:
            return False
    return True
``` ```python
# END SOLUTION
``` ```python

EXAMPLE 1
PROMPT:
You will be given a programming problem and will generate a correct Python function that matches the specification and passes all tests.

You will NOT return anything except for the function impl